<a href="https://colab.research.google.com/github/fauziardiantama/my-falcon-tuning/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Falcon 90M Training & Auto-Transfer

Notebook ini dikonfigurasi untuk menjalankan training pada Falcon 90M Instruct (Full FT atau LoRA) dan mengirimkan hasilnya ke komputer lokal menggunakan `croc`.

In [ ]:
# 1. Clone Repositori dan Pindah ke Direktori
import os
REPO_URL = 'https://github.com/fauziardiantama/my-falcon-tuning.git'
REPO_NAME = 'my-falcon-tuning'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# --- DIAGNOSTICS ---
print("--- SYSTEM DIAGNOSTICS ---")
!nvcc --version
!python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA available: {torch.cuda.is_available()}'); print(f'CUDA version: {torch.version.cuda}')"

# 2. Instalasi Requirements
TRAINING_METHOD = 'full' # 'full' atau 'lora'
REQ_FILE = 'requirements_lora.txt' if TRAINING_METHOD == 'lora' else 'requirements.txt'

print(f"--- INSTALLING DEPENDENCIES FOR {TRAINING_METHOD.upper()} ---")
# Instalasi standar untuk requirements umum
!pip install -q -r {REQ_FILE}

# Instalasi khusus library akselerasi dengan penanganan error yang lebih baik
if TRAINING_METHOD == 'full':
    print("--- INSTALLING MAMBA ACCELERATION KERNELS (CUDA 12.8 Optimized) ---")
    # Menggunakan --no-build-isolation agar pip menggunakan versi PyTorch/CUDA yang ada di sistem
    !MAX_JOBS=4 CAUSAL_CONV1D_FORCE_BUILD=TRUE pip install causal-conv1d>=1.5.0 --no-build-isolation
    !MAX_JOBS=4 MAMBA_FORCE_BUILD=TRUE pip install mamba-ssm --no-build-isolation

# 3. Instalasi Croc (File Transfer)
print("--- INSTALLING CROC ---")
!curl https://getcroc.schollz.com | bash

In [ ]:
# 4. Jalankan Script Training
TRAINING_SCRIPT = 'train_falcon_lora.py' if TRAINING_METHOD == 'lora' else 'train_falcon.py'
print(f"--- STARTING {TRAINING_METHOD.upper()} TRAINING ---")
!python {TRAINING_SCRIPT}

In [ ]:
# 5. Kirim Hasil Menggunakan Croc
OUTPUT_DIR = './falcon-90m-lora-adapter' if TRAINING_METHOD == 'lora' else './falcon-90m-fine-tuned'
print(f"--- SENDING {TRAINING_METHOD.upper()} RESULT VIA CROC ---")
print("Salin kode yang muncul di bawah ini untuk menerima file di komputer lokal:")
!croc send {OUTPUT_DIR}